# 再利用可能な埋め込み（Reusable Embeddings）

**学習目標**
1. 事前トレーニング済みの Kaggle Hub テキスト モジュールを使用して文ベクトルを生成する方法を学ぶ
1. 事前トレーニング済みの Kaggle Hub モジュールを Keras モデルに組み込む方法を学ぶ


## はじめに


このノートブックでは、タイトル データセットに含まれるタイトルの推定ソース（GitHub、TechCrunch、または The New York Times）を認識するテキスト モデルを実装します。

まず、テキストとラベルを読み込んで前処理を行い、最初の層に Kaggle Hub の事前トレーニング済みモジュールを持つ Keras モデルに入力できるようにします。この最初の層のおかげで、テキストをモデルに渡す前にトークン化したり整数化したりする必要がなくなります。事前トレーニング済みの層がそれを処理し、生のテキストを直接受け取ります。ただし、3 つの各クラスを 3 次元の基底ベクトルにワンホット（one-hot）エンコードする必要はあります。

次に、事前トレーニング済みの Kaggle Hub レイヤーから始まるシンプルな DNN モデルを構築、トレーニングし、比較します。

In [ ]:
import os
import warnings

import pandas as pd
from google.cloud import bigquery

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

%load_ext tensorboard
%load_ext google.cloud.bigquery


以下のセルの変数の値を置き換えてください。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT  # デフォルトは PROJECT
REGION = "us-central1"  # 実際の REGION に置き換えてください

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

## BigQuery からデータセットを作成

Hacker News のヘッドラインは、BigQuery の一般公開データセットとして利用可能です。[データセット](https://bigquery.cloud.google.com/table/bigquery-public-data:hacker_news.stories?tab=details) には、2006年10月のサイト開設から2015年10月までのすべてのヘッドラインが含まれています。

データセットのサンプルは以下のとおりです。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    url, title, score
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    LENGTH(title) > 10
    AND score > 10
    AND LENGTH(url) > 0
LIMIT 10

URL から新聞記事のソース（情報源）を取得するために、BigQuery で正規表現のパースを行いましょう。たとえば、URL が http://mobile.nytimes.com/.... の場合、<i>nytimes</i> だけを抽出します。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.'))[SAFE_OFFSET(1)] AS source,
    COUNT(title) AS num_articles
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.com$')
    AND LENGTH(title) > 10
GROUP BY
    source
ORDER BY num_articles DESC
  LIMIT 100

URL をパースしてソースを適切に取得できるようになったので、ソースとタイトルをまとめたデータセットを作成しましょう。これが機械学習用のラベル付きデータセットになります。

In [ ]:
regex = ".*://(.[^/]+)/"


sub_query = """
SELECT
    title,
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '{0}'), '.'))[SAFE_OFFSET(1)] AS source
    
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '{0}'), '.com$')
    AND LENGTH(title) > 10
""".format(
    regex
)


query = """
SELECT 
    LOWER(REGEXP_REPLACE(title, '[^a-zA-Z0-9 $.-]', ' ')) AS title,
    source
FROM
  ({sub_query})
WHERE (source = 'github' OR source = 'nytimes' OR source = 'techcrunch')
""".format(
    sub_query=sub_query
)

print(query)

機械学習のトレーニングでは通常、データセットをトレーニング用と評価用のデータセットに分割する必要があります（評価データセットに基づいてモデルや特徴量の選択を行う場合は、独立したテスト データセットも必要になることがあります）。ただし AutoML はこれらの分割方法を自動的に判断するため、ここでは手動で行う必要はありません。

In [ ]:
bq = bigquery.Client(project=PROJECT)
title_dataset = bq.query(query).to_dataframe()
title_dataset.head()

テキスト分類用の AutoML では、以下の要件を満たす必要があります。
* データセットが CSV 形式であること
* 最初の列が分類対象のテキスト、またはテキストへの GCS パスであること
* 最後の列がテキストのラベルであること

BigQuery から取得したデータセットはこれらの要件を満たしています。

In [ ]:
print(f"The full dataset contains {len(title_dataset)} titles")

3 つの各ラベルのサンプル数がほぼ同数であることを確認しましょう。

In [ ]:
title_dataset.source.value_counts()

最後に、現在メモリ上にあるデータをディスクに保存します。

完全なデータセットを含む CSV ファイルと、開発用に 1,000 件の記事のみを含む CSV ファイルを作成します。

**注:** 完全なデータセットで AutoML をトレーニングするには時間がかかる場合があるため、ツールの学習目的にはサンプル データセットを使用することをお勧めします。

In [ ]:
DATADIR = "./data/"

if not os.path.exists(DATADIR):
    os.makedirs(DATADIR)

In [ ]:
FULL_DATASET_NAME = "titles_full.csv"
FULL_DATASET_PATH = os.path.join(DATADIR, FULL_DATASET_NAME)

# ディスクに書き込む前にデータをシャッフルします。
title_dataset = title_dataset.sample(n=len(title_dataset))

title_dataset.to_csv(
    FULL_DATASET_PATH, header=False, index=False, encoding="utf-8"
)

## 再利用可能な埋め込みモデル

In [ ]:
import datetime
import os
import shutil

import keras
import pandas as pd
import tensorflow as tf
from keras.callbacks import EarlyStopping, TensorBoard
from keras.layers import Dense, Input, Lambda, TextVectorization
from keras.models import Model
from keras.utils import to_categorical
from tensorflow_hub import KerasLayer

print(tf.__version__)

In [ ]:
%matplotlib inline

まず、トレーニング済みモデルに関する情報が保存される場所と、データセットが配置されている場所を指定します。

In [ ]:
MODEL_DIR = f"gs://{BUCKET}/text_models"

## データセットの読み込み

前のラボと同様に、データセットは記事のタイトルと、それらの記事がどのソース（GitHub、TechCrunch、または The New York Times）から取得されたかを示すラベルで構成されています。

In [ ]:
ls $DATADIR

In [ ]:
DATASET_NAME = "titles_full.csv"
TITLE_SAMPLE_PATH = os.path.join(DATADIR, DATASET_NAME)
COLUMNS = ["title", "source"]

titles_df = pd.read_csv(TITLE_SAMPLE_PATH, header=None, names=COLUMNS)
titles_df.head()

データセットのバランスが適切であることを確認するために、ラベルごとのサンプル数をもう一度確認しましょう。

In [ ]:
titles_df.source.value_counts()

## ラベルの準備

このラボでは、モデルの最初の層として、英語用の事前トレーニング済み [Kaggle Hub 埋め込みモジュール](https://tfhub.dev/s?q=tf2%20embeddings%20text%20english) を使用します。これを行う直接的なメリットの 1 つは、Kaggle Hub の埋め込みモジュールが生テキストの処理を代行してくれることです。
これはまた、単語を表す整数のシーケンスの代わりに、モデルが生テキストを直接受け取ることができるようになることを意味します。

ただし前と同様に、ラベルをワンホット エンコードされたベクトルに前処理する必要があります。

In [ ]:
CLASSES = {"github": 0, "nytimes": 1, "techcrunch": 2}
N_CLASSES = len(CLASSES)

In [ ]:
def encode_labels(sources):
    classes = [CLASSES[source] for source in sources]
    one_hots = to_categorical(classes, num_classes=N_CLASSES)
    return one_hots

In [ ]:
encode_labels(titles_df.source[:4])

## トレーニング / テスト分割の準備

データをトレーニング分割とテスト分割に分けましょう。

In [ ]:
N_TRAIN = int(len(titles_df) * 0.95)

titles_train, sources_train = (
    titles_df.title[:N_TRAIN],
    titles_df.source[:N_TRAIN],
)

titles_valid, sources_valid = (
    titles_df.title[N_TRAIN:],
    titles_df.source[N_TRAIN:],
)

念のため、トレーニング分割とテスト分割において、クラスごとにほぼ同数のサンプルがあることを確認します。

実際にバランスが取れているため、モデルのパフォーマンスを測定する指標として正解率（accuracy）が適しています。

In [ ]:
sources_train.value_counts()

In [ ]:
sources_valid.value_counts()

それでは、モデルに入力する特徴量とラベルを作成しましょう。

In [ ]:
X_train, Y_train = titles_train.to_numpy(), encode_labels(sources_train)
X_valid, Y_valid = titles_valid.to_numpy(), encode_labels(sources_valid)

In [ ]:
X_train[:3]

In [ ]:
Y_train[:3]

## NNLM モデル

まず、[ニューラル確率的言語モデル（Neural Probabilistic Language Model）](http://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) を使用して事前トレーニングされた単語埋め込みを試します。Kaggle Hub には、生成されたベクトルを正規化する [nnlm-en-dim50-with-normalization](https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1) という 50 次元のモデルが用意されています。

### ラボ課題 1a: NNLM TF Hub モジュールを `KerasLayer` にインポートする

URL から読み込まれた Kaggle Hub モジュールは、Keras モデルのレイヤーとして使用できます。事前トレーニング済み埋め込み自体のパラメータをファインチューニングするのに十分なデータがあるため、事前トレーニング済み埋め込みを読み込む `KerasLayer` で `trainable=True` を設定します。

In [ ]:
NNLM = "https://tfhub.dev/google/nnlm-en-dim50/2"

nnlm_module = KerasLayer(
    # TODO
)

この Kaggle Hub 埋め込みは、文を渡すと単一の 50 次元ベクトルを生成することに注意してください。

### ラボ課題 1b: モジュールを使用して文文字列をエンコードする

In [ ]:
nnlm_module(
    tf.constant(
        [
            # TODO
        ]
    )
)

## Swivel モデル

次に、単語の共起行列を行列分解して単語埋め込みを作成するアルゴリズムである [Swivel](https://arxiv.org/abs/1602.02215) を使用して取得された単語埋め込みを試します。
Kaggle Hub には、事前トレーニング済みの 20 次元 Swivel モジュールである [gnews-swivel-20dim-with-oov](https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim-with-oov/1) がホストされています。

### ラボ課題 1c: Swivel TF Hub モジュールを `KerasLayer` にインポートする

In [ ]:
SWIVEL = "https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim-with-oov/1"

swivel_module = KerasLayer(
    # TODO
)

前の事前トレーニング済み埋め込みと同様に、文を渡すと単一のベクトルを出力します。

### ラボ課題 1d: モジュールを使用して文文字列をエンコードする

In [ ]:
swivel_module(
    tf.constant(
        [
            # TODO
        ]
    )
)

## モデルの構築

Keras Functional API を使用して `build_model` 関数を作成し、事前トレーニング済みの Kaggle Hub 埋め込みレイヤーを活用してテキスト分類用に設計された Keras モデルを構築します。これにより、トレーニングと評価が可能なニューラル ネットワーク アーキテクチャが動的に作成されます。


構築するモデルの内訳と仕組みは次のとおりです。

- 入力レイヤー（Input Layer）: モデルは、生のテキスト文字列を受け入れるように構成された Input レイヤーを定義することから始まります。これにより、最初に数直線や数値形式に変換することなく、モデルが生テキストを直接処理できるようになります。

- 埋め込みレイヤー（Embedding Layer）: 事前トレーニング済みの TensorFlow Hub モデルである `hub_module` は、Lambda レイヤーでラップされます。これは、TensorFlow Hub モジュールを Keras Functional API と互換性を持たせ、他の Keras レイヤーと同様に使用できるようにするために必要なステップです。`module_output_dim` パラメータは、`hub_module` が生成する埋め込みの次元数を指定します。

- 分類ヘッド（Classification Head）: 埋め込みレイヤーに続いて、2 つ（またはそれ以上追加する場合）の全結合（Dense）レイヤーがネットワークの分類部分を形成します。

モデルの組み立てとコンパイル: 入力レイヤーと出力レイヤーを指定して Model を作成します。その後、adam オプティマイザ、多クラス分類の標準である `categorical_crossentropy` を損失関数、`accuracy` を評価指標としてモデルをコンパイルします。
### ラボ課題 2: 事前トレーニング済みの TF Hub モジュールを Keras Sequential モデルの最初の層として組み込む

In [ ]:
def build_model(hub_module, module_output_dim, name):
    # 生のテキスト文字列用の入力レイヤーを定義
    # TODO

    # KerasTensor を出力できるように hub_module の呼び出しを Lambda レイヤーでラップ
    # TODO

    # Dense レイヤーを追加
    # TODO

    # 分類用の最終 Dense レイヤーを追加
    # TODO

    # モデルを作成
    model = Model(inputs=inputs, outputs=outputs, name=name)

    # モデルをコンパイル
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=False,
    )
    return model

トレーニング コードも `train_and_evaluate` 関数にラップしましょう。この関数は:
* トレーニング データと検証データ、コンパイル済みモデル自体、および `batch_size` を入力として受け取ります
* コンパイル済みモデルを最大 100 エポックでトレーニングし、検証損失が減少しなくなったときに Early Stopping（早期終了）を行います
* 学習曲線をプロットするのに役立つ `history` オブジェクトを返します

In [ ]:
def train_and_evaluate(train_data, val_data, model, batch_size=5000):
    X_train, Y_train = train_data

    model_dir = os.path.join(MODEL_DIR, model.name)
    if tf.io.gfile.exists(model_dir):
        tf.io.gfile.rmtree(model_dir)

    history = model.fit(
        X_train,
        Y_train,
        epochs=100,
        batch_size=batch_size,
        validation_data=val_data,
        callbacks=[EarlyStopping(patience=3), TensorBoard(model_dir)],
    )
    return history

## NNLM のトレーニング

In [ ]:
data = (X_train, Y_train)
val_data = (X_valid, Y_valid)

In [ ]:
nnlm_model = build_model(
    hub_module=nnlm_module, module_output_dim=50, name="nnlm"
)
nnlm_history = train_and_evaluate(data, val_data, nnlm_model)

In [ ]:
history = nnlm_history
pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

## Swivel のトレーニング

In [ ]:
swivel_model = build_model(
    hub_module=swivel_module, module_output_dim=20, name="swivel"
)

In [ ]:
swivel_history = train_and_evaluate(data, val_data, swivel_model)

In [ ]:
history = swivel_history
pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

## モデルの比較

静的なプロットも有用ですが、インタラクティブな可視化を使用すると、2 つのモデルのパフォーマンスをより直接的に比較できます。TensorBoard を使用すると、Swivel モデルと NNLM モデルの両方の指標を単一のインタラクティブなグラフ上にオーバーレイ表示できます。これにより、Swivel の方が高速にトレーニングされますが、NNLM の方がより少ないトレーニング エポックで優れた検証正解率に収束することが明確にわかります。これらの結果を確認するには、以下のコマンドで TensorBoard ダッシュボードを起動します。

In [ ]:
%tensorboard --logdir $MODEL_DIR --port 8008

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License